# Sample selection


### **Data needed**
1. Daily prices for all NYSE/AMEX listed securities for 1974-1986 and 2012-2024, with market cap
2. Quarterly Earnings Per Share - Excludig Extraordinary Items, Price Close - Quarter End, Earnings Announcement Data (for 1982-1987 and 2020-2025)



### **Data sources**

Wharton Research Data Services. "WRDS" wrds.wharton.upenn.edu, accessed 2026-06-04.

##### **CRSP Daily**

*Query parameters*
- primaryexch (Primary Exchange (A:AMEX, B:BATS, I:IEX, N:NYSE, Q:NASDAQ, R:NYSE ARCA, X:Unknown))
- securitynm (Security Name)
- securitytype (Security Type)
- permco (PERMNO)
- ticker (Ticker)
- dlyprc (Daily Price)
- dlycap (Daily Capitalization)
- dlyopen (Daily Open)
- dlyclose (Daily Close)
- dlylow (Daily Low)
- dlyhigh (Daily High)

*Files*
- ```CRSP_Daily_1974_1986.csv``` (2.09GB)         From 1974-01-01 to 1986-12-31
- ``CRSP_Daily_2012_2025.csv`` (469.0MB)                  From 2012-01-01 to 2024-12-31


##### **Compustat Quarterly**

*Query parameters*
- conm (Company Name)
- tic (Ticker Symbol)
- rdq (Report Date of Quarterly Earnings)
- fyearq (Fiscal Year)
- fqtr (Fiscal Quarter)
- prccq (Price Close - Quarter)
- epsfxq (Earnings Per Share (Diluted) - Excluding Extraordinary items)
- epspxq (Earnings Per Share (Basic) - Excluding Extraordinary Items)
- cshoq (Common Shares Outstanding)
- cshprq (Common Shares Used to Calculate Earnings Per Share - Basic)
- ajexq (Adjustment Factor (Company) - Cumulative by Ex-Date)

*Files*
- ``Compustat_Q_196103_198712.csv`` (48.2MB)    From 1961-03 to 1987-12
- ``Compustat_Q_199903_202512.csv`` (137.2MB)   From 1999-03 to 2025-12

##### **Mapping table**

*Query parameters*
- gvkey
- conm
- tic
- cusip
- LINKPRIM
- LIID
- LINKTYPE
- LPERMNO
- LPERMCO
- LINKDT
- LINKENDDT

*Files*
- ``Compustat_CRSP_link.csv`` (3.3MB)

### **Paper methodology**

"*Our sample includes 84,792 firm-quarters of data for NYSE/AMEX firms for 1974-86. Criteria for inclusion in the sample are the same as those used by FOS, who studied NYSE/AMEX firms for the period 1974-81. We require that the firm be listed on the CRSP daily files, and that the firm's earnings before extraordinary items and discontinued operations be available for at least ten consecutive quarters on Compustat. Our NYSE/AMEX sample includes only firms that appeared on any of the Compustat files released from 1982 through 1987*."

### **Replication**
We first load Compustat Quarterly dataset obtained from WRDS.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

dfQ = pd.read_csv("../data/WRDS/Compustat_Q_196103_198712.csv")
dfQ['datadate'] = pd.to_datetime(dfQ['datadate'])
dfQ.head()

,costat,curcdq,datafmt,indfmt,consol,tic,datadate,gvkey,conm,ajexq,fqtr,fyearq,rdq,cshoq,cshprq,epsfxq,epspxq,prccq
0,I,USD,STD,INDL,C,SHLDQ,1961-03-31,6307,SEARS HOLDINGS CORP,54.000000,1,1961,NaN,NaN,5.518,NaN,NaN,NaN
1,I,USD,STD,INDL,C,MHT.1,1961-03-31,6990,MANHATTAN INDUSTRIES INC,3.384872,1,1961,NaN,NaN,0.435,NaN,0.25,NaN
2,I,USD,STD,INDL,C,WIX1,1961-04-30,4979,GAMBLE-SKOGMO,1.030000,1,1961,NaN,NaN,2.652,NaN,NaN,NaN
3,I,USD,STD,INDL,C,KDT,1961-04-30,6303,KDT INDUSTRIES INC,7.731473,1,1961,NaN,NaN,0.800,NaN,0.15,NaN
4,A,USD,STD,INDL,C,VNO,1961-04-30,11220,VORNADO REALTY TRUST,40.927468,1,1961,NaN,NaN,1.311,NaN,NaN,NaN


Following Foster's (1977) methodology used by Bernard & Thomas (1989), we only shortlist firms that are present in the 1982-1987 Compustat files.

In [23]:
dfQ_slice = dfQ[(dfQ['datadate'] >= '1982-01-01') & (dfQ['datadate'] <= '1987-12-31')]
tickers_1982_1987 = dfQ_slice['tic'].dropna().unique()
print(f"Number of unique tickers in the 1982-1987 window: {len(tickers_1982_1987)}")

Number of unique tickers in the 1982-1987 window: 11004


Among these, we only keep firms that have at least ten consecutive earnings observations. We don't know if we should use epspxq (basic Earning per Share) instead of epsfxq (diluted EPS).


In [24]:
# Checking for 10 consecutive quarterly announcement for shortlisted firms
dfQ = dfQ.sort_values(by=["tic", "fyearq", "fqtr"])
dfQ["pidx"] = dfQ["fyearq"] * 4 + dfQ["fqtr"]

df_filtered = dfQ[dfQ["tic"].isin(tickers_1982_1987)].copy()
sub = df_filtered[df_filtered["epspxq"].notna()].copy() # epsfxq (85957 final) or epspxq (?? final)
sub["break"] = sub.groupby("tic")["pidx"].diff().ne(1)
sub["block"] = sub.groupby("tic")["break"].cumsum()

# 10 or more consecutive quarters
sizes = sub.groupby(["tic", "block"]).size().reset_index(name="count")
valid_firms = sizes[sizes["count"] >= 10]["tic"].unique()
print(f"{len(valid_firms)} shortlisted firms (10-consecutive quarters, ticker present in 1982-1987)")
dfQ.head()

# Shortlisted Compustat DF, without veryfing exchange or CRSP presence
valid_firms = dfQ[dfQ['tic'].isin(valid_firms)][['tic', 'gvkey']].drop_duplicates(subset=['tic'])

7695 shortlisted firms (10-consecutive quarters, ticker present in 1982-1987)


We then load CRSP Daily files, filtering firms not listed on NYSE/AMEX, following the authors method.

In [ ]:
# Loading data
dfD_NYSE = pd.read_csv("../data/WRDS/CRSP_Daily_1974_1986.csv")

# We only select NYSE/AMEX listed firms
dfD_NYSE = dfD_NYSE[dfD_NYSE['PrimaryExch'].isin(['A', 'N'])]

print(f"CRSD Daily observations (1974-1986): {len(dfD_NYSE)}\n   {len(dfD_NYSE['PERMNO'].unique())} firms (PERMNO)")
dfD_NYSE.head()

CRSD Daily observations (1974-1986): 8036197
   3959 firms (PERMNO)


,PERMNO,HdrCUSIP,PrimaryExch,SecurityNm,SecurityType,Ticker,PERMCO,DlyCalDt,DlyPrc,DlyCap,DlyClose,DlyLow,DlyHigh,DlyOpen
1238,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-02,57.875,324447.25,57.875,57.50,58.50,NaN
1239,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-03,61.250,343367.50,61.250,59.50,61.25,NaN
1240,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-04,59.750,334958.50,59.750,59.75,61.25,NaN
1241,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-07,58.250,326549.50,58.250,57.25,59.50,NaN
1242,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-08,56.875,318841.25,56.875,56.25,58.00,NaN


Compustat firms are identifyed with their unique *gvkey*. CRSP firms are identified with their *PERMNO*. We use the CRSP/Compustat linking table to match both shortlisted firms from Compustat to shortlisted firms present in CRSP Daily files.

In [ ]:
mapping = pd.read_csv("../data/WRDS/Compustat_CRSP_link.csv")

# Filtering link type
mapping_filtered = mapping[(mapping['LINKTYPE'].isin(['LC', 'LU'])) & (mapping['LINKPRIM'].isin(['P', 'C']))].copy()

# Time filter on links
mapping_filtered['LINKDT']  = pd.to_datetime(mapping_filtered['LINKDT'],  errors='coerce')
mapping_filtered['LINKENDDT'] = pd.to_datetime(mapping_filtered['LINKENDDT'], errors='coerce')

period_start, period_end = pd.Timestamp('1982-01-01'), pd.Timestamp('1987-12-31')

mapping_filtered = mapping_filtered[(mapping_filtered['LINKDT'].isna()  | (mapping_filtered['LINKDT']  <= period_end)) & (mapping_filtered['LINKENDDT'].isna() | (mapping_filtered['LINKENDDT'] >= period_start)) ]

mapping_filtered['gvkey'] = mapping_filtered['gvkey'].astype(str)
valid_firms['gvkey']      = valid_firms['gvkey'].astype(str)

# Joining gvkey Compustat with shortlisted PERMNO
matched_ids = pd.merge(
    valid_firms[['gvkey']],
    mapping_filtered[['gvkey', 'LPERMNO', 'LINKDT', 'LINKENDDT']],
    on='gvkey',
    how='inner'
).rename(columns={'LPERMNO': 'PERMNO'})

# Matching with NYSE/AMEX
crsp_permnos = dfD_NYSE['PERMNO'].unique()
final_ids = matched_ids[matched_ids['PERMNO'].isin(crsp_permnos)].copy()

# Handling duplicate: we keep those covering the largest period
final_ids = (
    final_ids
    .sort_values('LINKENDDT', ascending=False, na_position='first')
    .drop_duplicates(subset='gvkey', keep='first')
    [['gvkey', 'PERMNO']]
)

print(f"{len(final_ids)} unique gvkey-PERMNO pairs")


2732 unique gvkey-PERMNO pairs


These 2732 firms are our final datasets of interest: we applied all filters used by the researchers.

In [1]:
# Clean dataframes
dfQ['gvkey'] = dfQ['gvkey'].astype(str)
dfQ_final = dfQ.merge(final_ids, on='gvkey', how='inner')[
    ['gvkey', 'PERMNO', 'tic', 'datadate', 'fqtr', 'fyearq', 'epspxq', 'prccq', 'rdq']
]

dfD_NYSE = dfD_NYSE.sort_values(['PERMNO', 'DlyCalDt'])
dfD_NYSE['DlyReturns'] = dfD_NYSE.groupby('PERMNO')['DlyClose'].pct_change()


dfD_final = dfD_NYSE.merge(final_ids, on='PERMNO', how='inner')[
    ['gvkey', 'PERMNO', 'Ticker', 'DlyCalDt', 'DlyReturns', 'DlyCap']
]

dfQ_final = dfQ_final[(dfQ_final['datadate'] >= '1974-01-01') & (dfQ_final['datadate'] <= '1986-12-31')]
dfQ_final = dfQ_final[dfQ_final['epspxq'].notna()]
print(f"{len(dfQ_final)} firm-quarters")

dfD_NYSE.to_csv("data/dfD_NYSE.csv", index=False)
dfQ_final.to_csv("data/dfQ_final.csv", index=False)
dfD_final.to_csv("data/dfD_final.csv", index=False)

NameError: name 'dfQ' is not defined

We find a total of xxx firm-quarters. This is above the 84792 obtained in the paper, but we haven't fully cleaned them yet.